In [5]:
from google.colab import userdata
import os

# Securely extract the token from your Colab Secrets vault
# Make sure you added a secret named GITHUB_TOKEN in your left sidebar (Key icon)
token = userdata.get('GITHUB_TOKEN')

# Set the token as a temporary environment variable for this cell execution
os.environ['GITHUB_TOKEN'] = token

# Configure Git credentials
!git config --global user.name "kkowal22"
!git config --global user.email "kathy.kowal89@gmail.com"

# Clone using the environment variable to keep the token completely hidden
!git clone https://$GITHUB_TOKEN@github.com/kkowal22/FDA_recalls_analysis.git

Cloning into 'FDA_recalls_analysis'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 34 (delta 9), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 226.59 KiB | 2.12 MiB/s, done.
Resolving deltas: 100% (9/9), done.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Change directory into the cloned repository folder
%cd FDA_recalls_analysis

# 1. Grab your active notebook from the runtime environment using a wildcard
!cp /content/*.ipynb "FDA Class I Drug Recall — Administrative Latency & Ingestion Pipeline/"

# 2. Copy the three CSV data files from your Google Drive path
# Note: Since these uploaded successfully last time, Git will automatically skip them if they haven't changed
!cp "/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2013 to Dec 2018.csv" "FDA Class I Drug Recall — Administrative Latency & Ingestion Pipeline/"
!cp "/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2019 to Dec 2023.csv" "FDA Class I Drug Recall — Administrative Latency & Ingestion Pipeline/"
!cp "/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2024 to 12 July 2026.csv" "FDA Class I Drug Recall — Administrative Latency & Ingestion Pipeline/"

# Stage, commit, and push the changes
!git add .
!git commit -m "Upload notebook and raw CSV datasets from Colab/Drive"
!git push origin main

[Errno 2] No such file or directory: 'FDA_recalls_analysis'
/content/FDA_recalls_analysis
cp: cannot stat '/content/*.ipynb': No such file or directory
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [ ]:
%pip install pandas numpy altair statsmodels

import pandas as pd
import numpy as np
import altair as alt
import statsmodels.formula.api as smf

In [ ]:
# Step 1: Load and dedupe

# 1. Define files with the correct .csv extension
file_paths = [
    '/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2013 to Dec 2018.csv',
    '/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2019 to Dec 2023.csv',
    '/content/drive/MyDrive/FDA recall data analysis/Raw data/Class 1 Drug recalls Jan 2024 to 12 July 2026.csv'
]

# 2. Use pd.read_csv instead of pd.read_excel
df_list = [pd.read_csv(path) for path in file_paths]
df = pd.concat(df_list, ignore_index=True)

# 3. Export the combined raw data to a single CSV file
output_csv_path = 'Class 1 combined Jan 2013 to July 12 2026 CSV.csv'
df.to_csv(output_csv_path, index=False)
print ("-" * 60)
print(f"Successfully converted and saved: {output_csv_path}")
print ("-" * 60)

df['Recall Initiation Date'] = pd.to_datetime(df['Recall Initiation Date'], errors='coerce')
df['Center Classification Date'] = pd.to_datetime(df['Center Classification Date'], errors='coerce')

# Keep only recalls initiated on or after 01/01/2013 — drop anything earlier
cutoff = pd.Timestamp('2013-01-01')
df = df[df['Recall Initiation Date'] >= cutoff].copy()

df_events = df.drop_duplicates(subset=['Event ID']).copy()
print ("-" * 60)
print (f"Event Aggregation and Verification")
print ("-" * 60)
print(f"Event ID count: {len(df)}")
print(f"Unique Event ID count after dedup: {len(df_events)}")
print ("-" * 60)

------------------------------------------------------------
Successfully converted and saved: Class 1 combined Jan 2013 to July 12 2026 CSV.csv
------------------------------------------------------------
------------------------------------------------------------
Event Aggregation and Verification
------------------------------------------------------------
Event ID count: 1675
Unique Event ID count after dedup: 603
------------------------------------------------------------


In [ ]:
# Step 2 and 3: Compute core latency field and assign each event to a calendar year

df_events['Latency_Days'] = (
    df_events['Center Classification Date'] - df_events['Recall Initiation Date']
).dt.days
df_events['Latency_Days'] = df_events['Latency_Days'].clip(lower=0)
df_events['Year'] = df_events['Recall Initiation Date'].dt.year

In [ ]:
df_events['Report Date'] = pd.to_datetime(df_events['Report Date'], errors='coerce')
df_events['Report_Lag'] = (df_events['Report Date'] - df_events['Recall Initiation Date']).dt.days

PULL_DATE = pd.Timestamp('2026-07-12')
safety_margin = int(df_events['Report_Lag'].quantile(0.90))  # ~196 days
df_events['Reliable'] = df_events['Recall Initiation Date'] <= (PULL_DATE - pd.Timedelta(days=safety_margin))

In [ ]:
# Step 4: Calculate backlog-signal metrics per year
# a.Calculate median of latency days per year
# b.Calculate the 90th percentile of latency days per year
# c.Calculate the threshold % (eg. >60 or >90 days) of latency days per year

def pct_over(series, threshold):
    return (series.dropna() > threshold).mean() * 100

annual_metrics = df_events.groupby('Year').agg(
    total_recall_volume=('Event ID', 'nunique'),
    median_latency=('Latency_Days', 'median'),
    p90_latency=('Latency_Days', lambda x: np.percentile(x.dropna(), 90)),
    pct_over_60=('Latency_Days', lambda x: pct_over(x, 60)),
    pct_over_90=('Latency_Days', lambda x: pct_over(x, 90)),
).reset_index()

print ("-" * 60)
print(f"Annual Backlog Metrics")
print ("-" * 60)
print(annual_metrics.to_string(index=False))





------------------------------------------------------------
Annual Backlog Metrics
------------------------------------------------------------
 Year  total_recall_volume  median_latency  p90_latency  pct_over_60  pct_over_90
 2013                   65           141.0        299.0    98.461538    80.000000
 2014                   48           109.0        220.6    89.583333    64.583333
 2015                   51           140.0        358.0    86.274510    72.549020
 2016                   32           105.0        175.7    90.625000    65.625000
 2017                   43            76.0        136.0    65.116279    37.209302
 2018                   41            35.0         74.0    17.073171     7.317073
 2019                   36            26.0         61.0    11.111111     2.777778
 2020                   50            32.5        122.4    22.000000    14.000000
 2021                   58            35.0        115.9    29.310345    15.517241
 2022                   58         

In [ ]:
# Step 5: Build the queue-depth (open-caseload) time series
# For each month-end and quarter-end date T in the data range, count events where:
# Recall Initiation Date <= T and Center Classification Date > T

start = df_events['Recall Initiation Date'].min()
end = df_events['Recall Initiation Date'].max()
snapshots = pd.date_range(start=start, end=end, freq='ME')  # month-end dates

queue_depth = []
for snap in snapshots:
    open_count = (
        (df_events['Recall Initiation Date'] <= snap) &
        (df_events['Center Classification Date'] > snap)
    ).sum()
    queue_depth.append({'Snapshot': snap, 'Open_Events': open_count})

queue_df = pd.DataFrame(queue_depth)

print ("-" * 60)
print(f"Queue Depth Time Series")
print ("-" * 60)
print(queue_df.tail(12).to_string(index=False))

------------------------------------------------------------
Queue Depth Time Series
------------------------------------------------------------
  Snapshot  Open_Events
2025-06-30            2
2025-07-31            3
2025-08-31            5
2025-09-30            3
2025-10-31            4
2025-11-30            3
2025-12-31            6
2026-01-31            6
2026-02-28            3
2026-03-31            2
2026-04-30            3
2026-05-31            2


In [ ]:
# Step 5 Graph: Build the queue-depth (open-caseload) time series
# Chart: Median vs P90 latency by year
latency_long = annual_metrics.melt(
    id_vars='Year',
    value_vars=['median_latency', 'p90_latency'],
    var_name='Metric',
    value_name='Days'
)
latency_long['Metric'] = latency_long['Metric'].map({
    'median_latency': 'Median',
    'p90_latency': '90th percentile'
})

latency_chart = alt.Chart(latency_long).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('Days:Q', title='Latency (days)'),
    color=alt.Color('Metric:N', scale=alt.Scale(range=['#2a78d6', '#e34948']),
                     legend=alt.Legend(title=None, orient='top')),
    strokeDash=alt.StrokeDash('Metric:N', legend=None),
    tooltip=['Year:O', 'Metric:N', alt.Tooltip('Days:Q', format='.1f')]
).properties(
    title='Median vs 90th Percentile Latency by Year',
    width=650, height=320
)
latency_chart

KeyError: 'Data_Status'

In [ ]:
# Step 5 Graph: Build the queue-depth (open-caseload) time series
# Chart: % of recalls exceeding threshold, by year
pct_long = annual_metrics.melt(
    id_vars='Year',
    value_vars=['pct_over_60', 'pct_over_90'],
    var_name='Threshold',
    value_name='Percent'
)
pct_long['Threshold'] = pct_long['Threshold'].map({
    'pct_over_60': '> 60 days',
    'pct_over_90': '> 90 days'
})

pct_chart = alt.Chart(pct_long).mark_bar().encode(
    x=alt.X('Year:O', title='Year'),
    y=alt.Y('Percent:Q', title='% of recalls exceeding threshold'),
    color=alt.Color('Threshold:N', scale=alt.Scale(range=['#eda100', '#e34948']),
                     legend=alt.Legend(title=None, orient='top')),
    xOffset='Threshold:N',
    tooltip=['Year:O', 'Threshold:N', alt.Tooltip('Percent:Q', format='.1f')]
).properties(
    title='Share of Recalls Exceeding Latency Threshold, by Year',
    width=650, height=320
)
pct_chart

alt.Chart(...)

In [ ]:
# Step 5 graph: Chart for Build the queue-depth (open-caseload) time series
area = alt.Chart(queue_df).mark_area(
    line={'color': '#2a78d6', 'size': 2},
    color=alt.Gradient(
        gradient='linear',
        stops=[alt.GradientStop(color='white', offset=0),
               alt.GradientStop(color='#2a78d6', offset=1)],
        x1=1, x2=1, y1=1, y2=0
    ),
    opacity=0.25,
    interpolate='monotone'
).encode(
    x=alt.X('Snapshot:T', title='Month', axis=alt.Axis(format='%Y')),
    y=alt.Y('Open_Events:Q', title='Open events (month-end)'),
    tooltip=[alt.Tooltip('Snapshot:T', title='Month', format='%b %Y'),
             alt.Tooltip('Open_Events:Q', title='Open events')]
).properties(
    title='FDA Class I Recall — Monthly Queue Depth (2006-2026)',
    width=700, height=350
)

final_chart = area
final_chart

alt.Chart(...)

In [ ]:
# Step 6: Add year-level volume context above the existing latency chart
volume_chart = alt.Chart(annual_metrics).mark_bar(color='#888780').encode(
    x=alt.X('Year:O', title=None),
    y=alt.Y('total_recall_volume:Q', title='Total recalls (volume)'),
    tooltip=['Year:O', 'total_recall_volume:Q']
).properties(width=650, height=150)

volume_chart & latency_chart

alt.VConcatChart(...)

In [ ]:
# Step 7: Test the volume-backlog relationship properly (event-level)

# 1. Concurrent_Volume via searchsorted — O(n log n) instead of O(n^2) loop
WINDOW_DAYS = 45
dates = df_events['Recall Initiation Date'].values
order = np.argsort(dates)
sorted_dates = dates[order]
left = np.searchsorted(sorted_dates, sorted_dates - np.timedelta64(WINDOW_DAYS, 'D'), side='left')
right = np.searchsorted(sorted_dates, sorted_dates + np.timedelta64(WINDOW_DAYS, 'D'), side='right')
counts = np.empty(len(dates), dtype=int)
counts[order] = right - left - 1  # exclude self
df_events['Concurrent_Volume'] = counts
reg_df = df_events.dropna(subset=['Latency_Days', 'Concurrent_Volume', 'Year']).copy()

# 2. Raw correlation + full regression (coef, p-value, CI need statsmodels either way)
raw_corr = reg_df['Latency_Days'].corr(reg_df['Concurrent_Volume'])
full_model = smf.ols('Latency_Days ~ Concurrent_Volume + C(Year)', data=reg_df).fit()
coef, pval = full_model.params['Concurrent_Volume'], full_model.pvalues['Concurrent_Volume']
ci_low, ci_high = full_model.conf_int().loc['Concurrent_Volume']

# 3. Detrend both variables via group-mean subtraction (equivalent to Year fixed effects,
#    since a single categorical regressor's OLS residual = value minus its group mean)
reg_df['Detrended_Latency'] = reg_df['Latency_Days'] - reg_df.groupby('Year')['Latency_Days'].transform('mean')
reg_df['Detrended_Volume'] = reg_df['Concurrent_Volume'] - reg_df.groupby('Year')['Concurrent_Volume'].transform('mean')

print(f"Raw correlation: {raw_corr:.4f}")
print(f"Regression coef (controlling for Year): {coef:.4f}, p={pval:.4f}, 95% CI [{ci_low:.4f}, {ci_high:.4f}]")

# 4. Chart
points = alt.Chart(reg_df).mark_circle(size=40, opacity=0.5, color='#2a78d6').encode(
    x=alt.X('Detrended_Volume:Q', title='Concurrent volume (year effect removed)'),
    y=alt.Y('Detrended_Latency:Q', title='Latency (days, year effect removed)'),
    tooltip=[alt.Tooltip('Detrended_Volume:Q', format='.1f'),
             alt.Tooltip('Detrended_Latency:Q', format='.1f'), 'Year:O']
)
trend = points.transform_regression('Detrended_Volume', 'Detrended_Latency').mark_line(color='#e34948', strokeWidth=2)
(points + trend).properties(title='Concurrent Volume vs Detrended Latency (Backlog-vs-Load Test)', width=650, height=350)